# Coinbase REST API: extracting crypto currency data

The public endpoints of the [Coinbase API](https://docs.cdp.coinbase.com/) provide crypto currency prices without an API key. Currency pairs are written as `<coin>-<currency>`, e.g. `BTC-USD`.

Endpoints used in this notebook:
- `https://api.coinbase.com/v2/prices/<pair>/spot`: current spot price
- `https://api.exchange.coinbase.com/products/<pair>/stats`: 24h statistics
- `https://api.exchange.coinbase.com/products/<pair>/candles`: historical prices (OHLCV)

## Libraries and settings

In [ ]:
# Libraries
import os
import locale
import warnings

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Ignore warnings
warnings.filterwarnings("ignore")

# Settings
locale.setlocale(locale.LC_ALL, "")

# Define settings for graphics
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#1a1a19',
    'axes.facecolor': '#1a1a19',
    'axes.edgecolor': '#383835',
    'axes.labelcolor': '#c3c2b7',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.color': '#383835',
    'grid.linewidth': 0.6,
    'xtick.color': '#c3c2b7',
    'ytick.color': '#c3c2b7',
    'legend.frameon': False})

# Colors: one fixed color per coin, green/red for up/down
COINS = ['BTC', 'ETH', 'SOL', 'NEAR', 'DOGE']
COLORS = dict(zip(COINS, ['#3987e5', '#d95926', '#199e70', '#c98500', '#d55181']))
UP, DOWN = '#0ca30c', '#d03b3b'

# Base URLs of the Coinbase APIs
BASE_URL = 'https://api.coinbase.com/v2'
EXCHANGE_URL = 'https://api.exchange.coinbase.com'

# Current working directory
print(f'Current working directory: {os.getcwd()}')

## Get data

### A single request: the raw JSON response

In [ ]:
response = requests.get(f'{BASE_URL}/prices/BTC-USD/spot', timeout=10)
response.raise_for_status()

print('Status code:', response.status_code)
response.json()

### Current prices in USD, EUR and CHF

In [ ]:
currencies = ['USD', 'EUR', 'CHF']

rows = []
for coin in COINS:
    for cur in currencies:
        response = requests.get(f'{BASE_URL}/prices/{coin}-{cur}/spot', timeout=10)
        response.raise_for_status()
        rows.append(response.json()['data'])

# Long format: one row per coin and currency
df_prices = pd.DataFrame(rows)
df_prices['amount'] = df_prices['amount'].astype(float)
df_prices.head()

In [ ]:
# Wide format: one row per coin, one column per currency
df_spot = df_prices.pivot(index='base', columns='currency', values='amount')
df_spot = df_spot.loc[COINS, currencies]
df_spot.round(4)

### 24h market statistics

In [ ]:
rows = []
for coin in COINS:
    response = requests.get(f'{EXCHANGE_URL}/products/{coin}-USD/stats', timeout=10)
    response.raise_for_status()
    stats = response.json()
    stats['coin'] = coin
    rows.append(stats)

df_stats = pd.DataFrame(rows).set_index('coin')
df_stats = df_stats[['open', 'high', 'low', 'last', 'volume']].astype(float)

# Derived columns
df_stats['change_24h_%'] = (df_stats['last'] - df_stats['open']) / df_stats['open'] * 100
df_stats['range_24h_%'] = (df_stats['high'] - df_stats['low']) / df_stats['low'] * 100
df_stats['volume_usd_mio'] = df_stats['volume'] * df_stats['last'] / 1e6
df_stats

In [ ]:
# Styled market overview: gains in green, losses in red
def color_change(val):
    return f'color: {UP if val >= 0 else DOWN}; font-weight: bold'


def arrow(val):
    return f'▲ {val:+.2f}%' if val >= 0 else f'▼ {val:+.2f}%'


(df_stats[['last', 'change_24h_%', 'high', 'low', 'range_24h_%', 'volume_usd_mio']]
    .sort_values('volume_usd_mio', ascending=False)
    .style
    .format({'last': '${:,.4f}', 'high': '${:,.4f}', 'low': '${:,.4f}',
             'change_24h_%': arrow, 'range_24h_%': '{:.2f}%',
             'volume_usd_mio': '${:,.1f} M'})
    .map(color_change, subset=['change_24h_%'])
    .bar(subset=['volume_usd_mio'], color='#3987e5', vmin=0)
    .set_caption('Crypto market overview (last 24 hours, USD)'))

### Historical daily prices (candles)

In [ ]:
# Daily candles (granularity in seconds), the API returns roughly the last year
def get_candles(coin, granularity=86400):
    response = requests.get(f'{EXCHANGE_URL}/products/{coin}-USD/candles',
                            params={'granularity': granularity},
                            timeout=10)
    response.raise_for_status()
    df = pd.DataFrame(response.json(),
                      columns=['time', 'low', 'high', 'open', 'close', 'volume'])
    df['date'] = pd.to_datetime(df['time'], unit='s')
    df['coin'] = coin
    return df.drop(columns='time').sort_values('date')


df_candles = pd.concat([get_candles(coin) for coin in COINS], ignore_index=True)
print(df_candles.shape)
df_candles.tail()

In [ ]:
# Closing prices: one column per coin
df_close = df_candles.pivot(index='date', columns='coin', values='close')[COINS]

# Daily returns in %
df_returns = df_close.pct_change() * 100

df_returns.describe().round(2)

## Visualize data

In [ ]:
# 24h price change per coin
df_plot = df_stats['change_24h_%'].sort_values()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(df_plot.index, df_plot.values,
               color=[UP if v >= 0 else DOWN for v in df_plot.values],
               height=0.6)
ax.axvline(0, color='#c3c2b7', linewidth=0.8)
ax.bar_label(bars, labels=[arrow(v) for v in df_plot.values],
             padding=4, color='white', fontsize=9)
ax.set_xlim(min(df_plot.min(), 0) * 1.4 - 0.5, max(df_plot.max(), 0) * 1.4 + 0.5)
ax.set_xlabel('Change in %')
ax.set_title('Price change in the last 24 hours', loc='left', fontweight='bold')
ax.grid(axis='y', visible=False)
plt.tight_layout()
plt.show()

In [ ]:
# Performance of the last 90 days, indexed to 100
days = 90
df_perf = df_close.tail(days)
df_perf = df_perf / df_perf.iloc[0] * 100

fig, ax = plt.subplots(figsize=(10, 5))
for coin in COINS:
    ax.plot(df_perf.index, df_perf[coin], color=COLORS[coin], linewidth=2, label=coin)
    # Direct label at the end of each line
    ax.annotate(f'{coin} {df_perf[coin].iloc[-1] - 100:+.1f}%',
                xy=(df_perf.index[-1], df_perf[coin].iloc[-1]),
                xytext=(6, 0), textcoords='offset points',
                va='center', fontsize=9, color='white')
ax.axhline(100, color='#c3c2b7', linewidth=0.8, linestyle='--')
ax.set_ylabel('Index (start = 100)')
ax.set_title(f'Which coin performed best? Last {days} days', loc='left', fontweight='bold')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
ax.legend(loc='upper left', ncols=len(COINS))
plt.tight_layout()
plt.show()

In [ ]:
# Candlestick chart with trading volume for Bitcoin (last 60 days)
coin = 'BTC'
df_btc = df_candles[df_candles['coin'] == coin].tail(60)
up = df_btc['close'] >= df_btc['open']
colors = np.where(up, UP, DOWN)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True,
                               gridspec_kw={'height_ratios': [3, 1]})

# High-low wicks and open-close bodies
ax1.vlines(df_btc['date'], df_btc['low'], df_btc['high'], color=colors, linewidth=1)
ax1.bar(df_btc['date'], (df_btc['close'] - df_btc['open']).abs(),
        bottom=df_btc[['open', 'close']].min(axis=1),
        color=colors, width=0.7)

# 20-day moving average
ma = df_close[coin].rolling(20).mean().loc[df_btc['date']]
ax1.plot(df_btc['date'], ma, color='white', linewidth=1.5, label='20-day moving average')
ax1.set_ylabel('Price (USD)')
ax1.yaxis.set_major_formatter(lambda x, pos: f'${x:,.0f}')
ax1.set_title(f'{coin}-USD: daily candles (green = up day, red = down day)',
              loc='left', fontweight='bold')
ax1.legend(loc='upper left')

# Volume in a separate panel below
ax2.bar(df_btc['date'], df_btc['volume'], color=colors, width=0.7)
ax2.set_ylabel(f'Volume ({coin})')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of daily returns: do the coins move together?
corr = df_returns.corr()

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(COINS)), COINS)
ax.set_yticks(range(len(COINS)), COINS)
ax.grid(False)
for i in range(len(COINS)):
    for j in range(len(COINS)):
        val = corr.iloc[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                color='white' if val > 0.45 else 'black', fontsize=10)
fig.colorbar(im, ax=ax, label='Correlation')
ax.set_title('Correlation of daily returns', loc='left', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Risk vs. return: annualized volatility vs. total return over the period
summary = pd.DataFrame({
    'total_return_%': (df_close.iloc[-1] / df_close.iloc[0] - 1) * 100,
    'volatility_%': df_returns.std() * np.sqrt(365)})

fig, ax = plt.subplots(figsize=(8, 5))
for coin, row in summary.iterrows():
    ax.scatter(row['volatility_%'], row['total_return_%'], s=160,
               color=COLORS[coin], edgecolor='#1a1a19', linewidth=2, zorder=3)
    ax.annotate(coin, (row['volatility_%'], row['total_return_%']),
                xytext=(10, 0), textcoords='offset points',
                va='center', color='white', fontsize=10)
ax.axhline(0, color='#c3c2b7', linewidth=0.8, linestyle='--')
ax.margins(x=0.15, y=0.12)
ax.set_xlabel('Annualized volatility in %')
ax.set_ylabel('Total return in %')
start, end = df_close.index[0], df_close.index[-1]
ax.set_title(f'Risk vs. return ({start:%d %b %Y} to {end:%d %b %Y})',
             loc='left', fontweight='bold')
plt.tight_layout()
plt.show()

summary.round(1)

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')